# 🥈 Bronze → Silver
Loads all 1480 CSVs, cleans and validates them, writes to a Silver Delta table.

**Output:** `rankrangers_project_data.silver.mhcet_allotments`

## Cell 1 — Config

In [ ]:
CSV_ROOT   = '/Volumes/rankrankers_project_data/pdf/cet_raw_pdfs/data/mahacet_cutoffs_2025_csv'
CATALOG    = 'rankrangers_project_data'
SCHEMA     = 'silver'
TABLE      = 'mhcet_allotments'
FULL_TABLE = f'{CATALOG}.{SCHEMA}.{TABLE}'

print(f'Source : {CSV_ROOT}')
print(f'Target : {FULL_TABLE}')

## Cell 2 — Create Schema

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}')
print(f'✅ Schema {CATALOG}.{SCHEMA} ready')

## Cell 3 — Load all CSVs into Bronze Spark DataFrame

In [ ]:
from pyspark.sql import functions as F

df_bronze = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .option('encoding', 'UTF-8') \
    .csv(f'{CSV_ROOT}/**/*.csv')

# Add source file path for traceability
df_bronze = spark.read \
    .option('header', True) \
    .option('inferSchema', True) \
    .option('encoding', 'UTF-8') \
    .csv(f'{CSV_ROOT}/**/*.csv') \
    .withColumn('_source_file', F.input_file_name())

print(f'📦 Bronze rows loaded : {df_bronze.count():,}')
print(f'📋 Columns           : {len(df_bronze.columns)}')
df_bronze.printSchema()

## Cell 4 — Clean & Standardise

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType

# ── Step 1: Drop VACANT rows ─────────────────────────────────────────────────
df = df_bronze.filter(
    (F.col('is_vacant') == False) &
    (F.col('application_id') != 'VACANT')
)
print(f'After removing VACANT rows : {df.count():,}')

# ── Step 2: Drop rows with no score or no application_id ─────────────────────
df = df.filter(
    F.col('mhtcet_score').isNotNull() &
    F.col('application_id').isNotNull() &
    F.col('application_id').rlike(r'^EN\d{8}$')
)
print(f'After removing invalid app IDs : {df.count():,}')

# ── Step 3: Clean candidate_category ─────────────────────────────────────────
# Strip special chars: OBC$#, SC/DEF1, NT 2 (NT-C) → OBC, SC, NT2
def clean_category(col):
    return (
        F.when(col.rlike('(?i)OPEN'), 'OPEN')
         .when(col.rlike('(?i)EWS'), 'EWS')
         .when(col.rlike('(?i)SBC|SEBC'), 'SBC')
         .when(col.rlike('(?i)OBC'), 'OBC')
         .when(col.rlike('(?i)SC'), 'SC')
         .when(col.rlike('(?i)ST'), 'ST')
         .when(col.rlike('(?i)VJ|DT'), 'VJ/DT')
         .when(col.rlike('(?i)NT.?1|NT.?A'), 'NT1')
         .when(col.rlike('(?i)NT.?2|NT.?B|NT.?C'), 'NT2')
         .when(col.rlike('(?i)NT.?3|NT.?D'), 'NT3')
         .when(col.rlike('(?i)PWD'), 'PWD')
         .when(col.rlike('(?i)DEF'), 'DEFENCE')
         .otherwise(F.regexp_replace(col, r'[$#@\(\)/0-9]', '').cast('string'))
    )

df = df.withColumn('clean_category', clean_category(F.col('candidate_category')))

# ── Step 4: Resolve gender from seat_type prefix ──────────────────────────────
# G prefix = General (Male eligible), L prefix = Ladies only
df = df.withColumn(
    'seat_gender',
    F.when(F.col('seat_type').startswith('L'), 'F')
     .when(F.col('seat_type').startswith('G'), 'M')
     .when(F.col('seat_type') == 'AI', 'AI')
     .when(F.col('seat_type') == 'MI', 'MI')
     .otherwise('M')  # PWDR, DEFR default to M
)

# ── Step 5: CAP round as integer ──────────────────────────────────────────────
df = df.withColumn(
    'cap_round_num',
    F.when(F.col('cap_round') == 'CAP-I',   1)
     .when(F.col('cap_round') == 'CAP-II',  2)
     .when(F.col('cap_round') == 'CAP-III', 3)
     .when(F.col('cap_round') == 'CAP-IV',  4)
     .otherwise(None)
)

# ── Step 6: Cast numeric columns ─────────────────────────────────────────────
df = df \
    .withColumn('mhtcet_score',   F.col('mhtcet_score').cast(FloatType())) \
    .withColumn('merit_no',       F.col('merit_no').cast(IntegerType())) \
    .withColumn('sr_no',          F.col('sr_no').cast(IntegerType())) \
    .withColumn('sanction_intake',F.col('sanction_intake').cast(IntegerType())) \
    .withColumn('cap_seats',      F.col('cap_seats').cast(IntegerType())) \
    .withColumn('minority_seats', F.col('minority_seats').cast(IntegerType())) \
    .withColumn('ai_seats',       F.col('ai_seats').cast(IntegerType()))

# ── Step 7: Drop rows where score is still null after cast ────────────────────
df = df.filter(F.col('mhtcet_score').isNotNull())

# ── Step 8: Add load timestamp ────────────────────────────────────────────────
df = df.withColumn('_load_ts', F.current_timestamp())

print(f'✅ Silver rows after cleaning: {df.count():,}')
display(df.limit(5))

## Cell 5 — Data Quality Report

In [ ]:
from pyspark.sql import functions as F

total = df.count()
print(f'📊 DATA QUALITY REPORT')
print(f'={'*50)
print(f'Total clean rows        : {total:,}')

# Null checks
for col in ['mhtcet_score','gender','seat_type','clean_category','institute_name','branch_name']:
    nulls = df.filter(F.col(col).isNull() | (F.col(col) == '')).count()
    pct = round(nulls/total*100, 2)
    status = '✅' if nulls == 0 else '⚠️ '
    print(f'{status} {col:25s}: {nulls:,} nulls ({pct}%)')

print(f'\n📦 Rows per CAP round:')
display(df.groupBy('cap_round','cap_round_num').count().orderBy('cap_round_num'))

print(f'\n🏷️  Category distribution:')
display(df.groupBy('clean_category').count().orderBy(F.col('count').desc()))

print(f'\n⚠️  Validation warnings summary:')
warned = df.filter(
    F.col('_validation_warnings').isNotNull() &
    (F.col('_validation_warnings') != '')
)
print(f'Rows with warnings: {warned.count():,} / {total:,}')
display(warned.groupBy('_validation_warnings').count().orderBy(F.col('count').desc()).limit(10))

## Cell 6 — Write Silver Delta Table

In [ ]:
df.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', True) \
    .saveAsTable(FULL_TABLE)

count = spark.table(FULL_TABLE).count()
print(f'✅ Written to {FULL_TABLE}')
print(f'   Rows: {count:,}')

## Cell 7 — Verify

In [ ]:
df_check = spark.sql(f'''
    SELECT cap_round, cap_round_num, clean_category, seat_gender,
           COUNT(*) as rows,
           ROUND(MIN(mhtcet_score),2) as min_score,
           ROUND(MAX(mhtcet_score),2) as max_score
    FROM {FULL_TABLE}
    GROUP BY cap_round, cap_round_num, clean_category, seat_gender
    ORDER BY cap_round_num, clean_category
''')
display(df_check)